In [1]:
import lfox
import lfox.lattice as lat
import jax
import jax.numpy as jnp

import numpy as np

In [2]:
np.ones(4)

array([1., 1., 1., 1.])

In [3]:
d = 2
L = 1024

In [4]:
MyLat = lat.SquareLattice(dims=((L,)*d))
MyLat

In [7]:
phi_field.nn_field(1)

Array([[1., 0., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       ...,
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.],
       [1., 1., 1., ..., 1., 1., 1.]], dtype=float32)

In [6]:
phi_field = lat.LatticeField(MyLat)
phi_field.field = np.ones_like(phi_field.field)
phi_field.field[0,0] = 0.
phi_field.field = jnp.array(phi_field.field)
print(phi_field.field, phi_field)

[[0. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 ...
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]
 [1. 1. 1. ... 1. 1. 1.]] <lfox.lattice.LatticeField object at 0x11ed01ed0>


I0000 00:00:1695591269.806074       1 tfrt_cpu_pjrt_client.cc:349] TfrtCpuClient created.


In [8]:
phi_field.nn((3,0), 0)

((4, 0), 1.0)

In [1]:
import inspect

In [2]:
dd = {'a': 3, 'b': 4.0, 'c': -1.1 }

def f(a,b):
    return 3*a-b

print(f(3,4.0))
print(f(**dd))

5.0


TypeError: f() got an unexpected keyword argument 'c'

In [3]:
sig = inspect.signature(f)
print(sig)
print(sig.parameters['b'])

for k in dd.keys():
    print(k in sig.parameters)

def safe_call_f(d):
    f_sig = inspect.signature(f)
    call = {}
    for k in d.keys():
        if k in f_sig.parameters:
            call[k] = d[k]

    return f(**call)

safe_call_f(dd)

(a, b)
b
True
True
False


5.0

In [4]:
for k, v in sig.parameters.items():
    print(k,v)

a a
b b


In [10]:
list(sig.parameters).index('a')

0

In [79]:
@jax.jit
def action_lfox(F):
    S = 0.0
    phi = F.field
    for i in range(L):
        for j in range(L):
            s = (i,j)
            for ax in range(d):
                s_nn, bc_mul = F.nn(s, ax)
#                S -= 2 * phi_field.field[s] * phi_field.nn(s, ax)
                S -= 2 * phi[s] * phi[s_nn] * bc_mul

    phi2 = phi_field.field**2
    S += jnp.sum(phi2)
    S += jnp.sum( (phi2-1)**2)

    return S

In [80]:
@jax.jit
def act2_lfox(F):
    S = 0.0
    phi = F.field
    for ax in range(d):
        S -= 2 * jnp.sum(phi * F.nn_field(ax))
    phi2 = phi**2
    S += jnp.sum(phi2)
    S += jnp.sum( (phi2-1)**2 )

    return S

In [81]:
Z = np.array(np.meshgrid(np.arange(2), np.arange(2), np.arange(2))).T.reshape(-1,3)
print(Z.shape, Z)
I = np.ones((2,2,2))
print(I.shape, I[Z[0]].shape)
np.array([ I[tuple(Zi)]  for Zi in Z ])

(8, 3) [[0 0 0]
 [0 1 0]
 [1 0 0]
 [1 1 0]
 [0 0 1]
 [0 1 1]
 [1 0 1]
 [1 1 1]]
(2, 2, 2) (3, 2, 2)


array([1., 1., 1., 1., 1., 1., 1., 1.])

In [82]:
@jax.jit
def action(phi):
    S = 0.0
    for ax in range(d):
        S -= 2*jnp.sum(phi * jnp.roll(phi, shift=1, axis=ax))
    phi2 = phi**2
    S += jnp.sum(phi2)
    S += jnp.sum( (phi2-1)**2 )
    return S

@jax.jit
def act2(phi):
    S = 0.0
    for i in range(L):
        for j in range(L):
            i_nn = (i+1) % L
            j_nn = (j+1) % L
            S -= 2 * phi[i,j] * phi[i_nn,j]
            S -= 2 * phi[i,j] * phi[i,j_nn]
    phi2 = phi**2
    S += jnp.sum(phi2)
    S += jnp.sum( (phi2-1)**2 )
    return S

@jax.jit
def act3(phi):
    S = 0.0
    all_sites = np.array(np.meshgrid(*[np.arange(L) for _ in range(d)]).T.reshape(-1,d))

phi_ones = np.ones((L,)*d)
       

In [93]:
%time action(phi_ones)

CPU times: user 8.29 ms, sys: 1.41 ms, total: 9.7 ms
Wall time: 4.06 ms


Array(-3145728., dtype=float32)

In [54]:
%time act2(phi_ones)

KeyboardInterrupt: 

In [43]:
%time action_lfox(phi_field)

6.75 µs ± 266 ns per loop (mean ± std. dev. of 7 runs, 100,000 loops each)


In [94]:
%time act2_lfox(phi_field)

CPU times: user 8.65 ms, sys: 1.97 ms, total: 10.6 ms
Wall time: 4.05 ms


Array(-3145720., dtype=float32)

In [71]:
phi_rand = np.random.rand(16,16)


In [73]:
print(action(phi_rand))
print(act2(phi_rand))

-70.96844
-70.96853


In [54]:
dimtest = (2,2,4)
jnp.meshgrid(*[jnp.arange(L) for L in dimtest])

[Array([[[0, 0, 0, 0],
         [1, 1, 1, 1]],
 
        [[0, 0, 0, 0],
         [1, 1, 1, 1]]], dtype=int32),
 Array([[[0, 0, 0, 0],
         [0, 0, 0, 0]],
 
        [[1, 1, 1, 1],
         [1, 1, 1, 1]]], dtype=int32),
 Array([[[0, 1, 2, 3],
         [0, 1, 2, 3]],
 
        [[0, 1, 2, 3],
         [0, 1, 2, 3]]], dtype=int32)]

In [53]:
jnp.meshgrid(*((jnp.arange(3),)*3))

[Array([[[0, 0, 0],
         [1, 1, 1],
         [2, 2, 2]],
 
        [[0, 0, 0],
         [1, 1, 1],
         [2, 2, 2]],
 
        [[0, 0, 0],
         [1, 1, 1],
         [2, 2, 2]]], dtype=int32),
 Array([[[0, 0, 0],
         [0, 0, 0],
         [0, 0, 0]],
 
        [[1, 1, 1],
         [1, 1, 1],
         [1, 1, 1]],
 
        [[2, 2, 2],
         [2, 2, 2],
         [2, 2, 2]]], dtype=int32),
 Array([[[0, 1, 2],
         [0, 1, 2],
         [0, 1, 2]],
 
        [[0, 1, 2],
         [0, 1, 2],
         [0, 1, 2]],
 
        [[0, 1, 2],
         [0, 1, 2],
         [0, 1, 2]]], dtype=int32)]

In [51]:
roll_test = np.random.rand(3,3)
print(roll_test)

coords_x, coords_y = np.meshgrid(np.arange(3), np.arange(3), indexing='ij')
print(coords_x, coords_y)

# Roll and apply BCs
roll_fwd = jnp.roll(roll_test, shift=1, axis=0)
print(jnp.ones_like(roll_fwd))

roll_fwd = roll_fwd.at[0,:].set(-1*roll_fwd[0,:])
print(roll_fwd)

print(np.roll(coords_x, shift=1, axis=0))
c, wind = np.divmod(coords_x+1, 3)
print(c, wind, (-1)**wind)
print((-1)**wind * jnp.roll(roll_test, shift=1, axis=0))

[[0.378624   0.5377509  0.84884542]
 [0.82509026 0.18185727 0.9154894 ]
 [0.54562297 0.95684539 0.56011135]]
[[0 0 0]
 [1 1 1]
 [2 2 2]] [[0 1 2]
 [0 1 2]
 [0 1 2]]
[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]
[[-0.54562294 -0.9568454  -0.56011134]
 [ 0.378624    0.5377509   0.8488454 ]
 [ 0.8250902   0.18185727  0.9154894 ]]
[[2 2 2]
 [0 0 0]
 [1 1 1]]
[[0 0 0]
 [0 0 0]
 [1 1 1]] [[1 1 1]
 [2 2 2]
 [0 0 0]] [[-1 -1 -1]
 [ 1  1  1]
 [ 1  1  1]]
[[-0.54562294 -0.9568454  -0.56011134]
 [ 0.378624    0.5377509   0.8488454 ]
 [ 0.8250902   0.18185727  0.9154894 ]]
